# 01 — 核心查询循环验证

逐步验证以下模块能否协同工作：

1. `permissions/` — 规则解析 + 权限决策
2. `tools/bash.py` — BashTool 执行
3. `core/query.py` — 流式循环 + 工具调用

**运行前**：确保 `.env` 文件已配置 `ANTHROPIC_API_KEY`。

## 0. 环境初始化

In [7]:
import sys
import os
from pathlib import Path

project_root = Path(".").resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(project_root / ".env")

api_key = os.getenv("ANTHROPIC_API_KEY")
model_id = os.getenv("MODEL_ID", "claude-sonnet-4-6")

assert api_key, "未找到 ANTHROPIC_API_KEY，请在项目根目录创建 .env 文件"
print(f"API Key 已加载（前缀：{api_key[:12]}...）")
print(f"使用模型：{model_id}")

API Key 已加载（前缀：sk-0MBKxPwwk...）
使用模型：claude-sonnet-4-6


## 1. 验证权限规则解析

In [8]:
from permissions.rules import parse_rule, rule_to_string

cases = [
    "Bash",
    "Bash(git status)",
    "Bash(python -c \"print\\(1\\)\")",
    "Bash(*)",
    "Bash()",
]

for s in cases:
    result = parse_rule(s)
    roundtrip = rule_to_string(result)
    print(f"输入:     {s!r}")
    print(f"解析:     {result}")
    print(f"序列化:   {roundtrip!r}")
    print()

输入:     'Bash'
解析:     tool_name='Bash' rule_content=None
序列化:   'Bash'

输入:     'Bash(git status)'
解析:     tool_name='Bash' rule_content='git status'
序列化:   'Bash(git status)'

输入:     'Bash(python -c "print\\(1\\)")'
解析:     tool_name='Bash' rule_content='python -c "print(1)"'
序列化:   'Bash(python -c "print\\(1\\)")'

输入:     'Bash(*)'
解析:     tool_name='Bash' rule_content=None
序列化:   'Bash'

输入:     'Bash()'
解析:     tool_name='Bash' rule_content=None
序列化:   'Bash'



## 2. 验证权限决策流水线

Jupyter 内置 event loop，直接用 `await` 而非 `asyncio.run()`。

In [9]:
from permissions.manager import PermissionManager, PermissionContext
from permissions.rules import PermissionMode
from tools.bash import BashTool

bash = BashTool()

# 场景 1：bypass 模式，应直接允许
ctx = PermissionContext(mode=PermissionMode.BYPASS)
decision = await PermissionManager(ctx).check(bash, {"command": "ls"})
print(f"bypass 模式:          {decision}")

# 场景 2：deny 规则命中，应直接拒绝
ctx = PermissionContext.from_rule_strings(deny=["Bash"])
decision = await PermissionManager(ctx).check(bash, {"command": "ls"})
print(f"deny 规则命中:        {decision}")

# 场景 3：allow 规则命中，应允许
ctx = PermissionContext.from_rule_strings(mode=PermissionMode.DEFAULT, allow=["Bash"])
decision = await PermissionManager(ctx).check(bash, {"command": "ls"})
print(f"allow 规则命中:       {decision}")

# 场景 4：default 模式无规则，应返回 ask
ctx = PermissionContext(mode=PermissionMode.DEFAULT)
decision = await PermissionManager(ctx).check(bash, {"command": "rm -rf /"})
print(f"default 模式（无规则）: {decision}")

bypass 模式:          behavior='allow' reason='权限模式 bypassPermissions 允许所有工具'
deny 规则命中:        behavior='deny' reason='Bash 被拒绝规则禁止使用'
allow 规则命中:       behavior='allow' reason='allow 规则匹配：Bash'
default 模式（无规则）: behavior='ask' reason='需要用户确认是否允许 Bash 执行' rule=None


## 3. 验证 BashTool 本地执行

In [10]:
from tools.bash import BashTool
from tools.base import ToolUseContext
from permissions.manager import PermissionManager, PermissionContext
from permissions.rules import PermissionMode

bash = BashTool(timeout=10)
ctx = ToolUseContext(
    permission_manager=PermissionManager(PermissionContext(mode=PermissionMode.BYPASS)),
    model=model_id,
    tools=[bash],
    session_id="notebook-test",
)

async def run_bash(command: str):
    print(f"$ {command}")
    async for chunk in bash.execute({"command": command}, ctx):
        print(chunk, end="")
    print()

await run_bash("echo Hello from BashTool")
await run_bash("python --version")
await run_bash("ls ../")

$ echo Hello from BashTool
Hello
from
BashTool

$ python --version
Python 3.13.13

$ ls ../


    Ŀ¼: D:\github_use\claude-code-py


Mode                 LastWriteTime         Length Name                                                                 
----                 -------------         ------ ----                                                                 
d-----         2026/6/21     17:07                .venv                                                                
d-----         2026/6/21     16:58                api                                                                  
d-----         2026/6/21     20:44                core                                                                 
d-----         2026/6/21     20:55                docs                                                                 
d-----         2026/6/21     21:13                notebooks                                                            
d-----         2026/6/21   

## 4. 验证核心查询循环（无工具，纯对话）

In [11]:
from core.query import QueryParams, query, TextDeltaEvent, MessageCompleteEvent
from permissions.manager import PermissionManager, PermissionContext
from permissions.rules import PermissionMode

params = QueryParams(
    messages=[{"role": "user", "content": "用一句话介绍你自己"}],
    system_prompt="你是一个简洁的助手，回答不超过两句话。",
    model=model_id,
    api_key=api_key,
)
mgr = PermissionManager(PermissionContext(mode=PermissionMode.BYPASS))

print("模型回复：", end="", flush=True)
async for event in query(params, tools=[], permission_manager=mgr):
    if isinstance(event, TextDeltaEvent):
        print(event.text, end="", flush=True)
    elif isinstance(event, MessageCompleteEvent):
        print(f"\n\n[stop_reason={event.stop_reason}, tokens={event.usage}]")

模型回复：我是一个由Anthropic开发的AI助手Claude，致力于以简洁、准确和有帮助的方式回答您的问题。

[stop_reason=end_turn, tokens={'input_tokens': 39, 'output_tokens': 45, 'cache_read_input_tokens': 0}]


## 5. 验证完整工具调用循环（核心）

In [12]:
from core.query import (
    QueryParams, query,
    StreamRequestStartEvent, TextDeltaEvent,
    ToolUseEvent, ToolResultEvent, MessageCompleteEvent,
)
from tools.bash import BashTool
from permissions.manager import PermissionManager, PermissionContext
from permissions.rules import PermissionMode

bash = BashTool(timeout=15)
mgr = PermissionManager(PermissionContext(mode=PermissionMode.BYPASS))

params = QueryParams(
    messages=[{
        "role": "user",
        "content": "用 Bash 工具列出当前目录下的 .py 文件，然后告诉我有几个。"
    }],
    system_prompt="你是一个编程助手，可以使用 Bash 工具执行命令。",
    model=model_id,
    api_key=api_key,
    max_turns=5,
)

print("=== 事件流 ===")
async for event in query(params, tools=[bash], permission_manager=mgr):
    if isinstance(event, StreamRequestStartEvent):
        print("\n--- 新一轮 API 请求 ---")
    elif isinstance(event, TextDeltaEvent):
        print(event.text, end="", flush=True)
    elif isinstance(event, ToolUseEvent):
        print(f"\n[工具调用] {event.tool_name}({event.tool_input})")
    elif isinstance(event, ToolResultEvent):
        preview = event.content[:200].replace("\n", "\\n")
        print(f"[工具结果] is_error={event.is_error} | {preview}")
    elif isinstance(event, MessageCompleteEvent):
        print(f"\n[完成] stop_reason={event.stop_reason}")

=== 事件流 ===

--- 新一轮 API 请求 ---

[工具调用] Bash({'command': 'find . -maxdepth 1 -name "*.py" -type f'})

[完成] stop_reason=tool_use
\n\n[退出码: 1]\nr=False | �Ҳ����ļ� - *.py

--- 新一轮 API 请求 ---
当前目录下 **没有找到任何 `.py` 文件**（即 0 个）。

`find` 命令在当前目录（`maxdepth 1` 限制只查一层）中搜索所有 `.py` 文件，结果为空，退出码为 1，说明当前目录下不存在任何 Python 源文件。

如果你想在**子目录中也一并搜索**，可以去掉 `-maxdepth 1` 限制，我可以再帮你执行一次～
[完成] stop_reason=end_turn


## 6. 验证 FileReadTool

In [13]:
from tools.file_read import FileReadTool
from tools.base import ToolUseContext
from permissions.manager import PermissionManager, PermissionContext
from permissions.rules import PermissionMode
from importlib import reload
import tools.file_read
reload(tools.file_read)
from tools.file_read import FileReadTool

read_tool = FileReadTool()
ctx = ToolUseContext(
    permission_manager=PermissionManager(PermissionContext(mode=PermissionMode.BYPASS)),
    model=model_id,
    tools=[read_tool],
    session_id="test",
)

async def test_read(path, **kwargs):
    label = f"{path}" + (f" {kwargs}" if kwargs else "")
    print(f"\n=== {label} ===")
    async for chunk in read_tool.execute({"file_path": path, **kwargs}, ctx):
        print(chunk, end="")

# 场景 1：读取本文件前 10 行
await test_read("../tools/file_read.py", start_line=1, end_line=10)

# 场景 2：文件不存在
await test_read("../tools/不存在.py")

# 场景 3：行范围超出文件
await test_read("../tools/file_read.py", start_line=999, end_line=1000)


=== ../tools/file_read.py {'start_line': 1, 'end_line': 10} ===
1	"""文件读取工具，读取本地文件内容并返回带行号的文本。"""
2	
3	from __future__ import annotations
4	
5	import os
6	from pathlib import Path
7	from typing import Any, AsyncIterator
8	
9	from tools.base import BaseTool, ToolUseContext
10	

=== ../tools/不存在.py ===
文件不存在：..\tools\不存在.py

=== ../tools/file_read.py {'start_line': 999, 'end_line': 1000} ===
指定行范围 [999, 1000] 无内容（文件共 111 行）。


## 7. 验证 FileEditTool

In [14]:
import importlib
import tools.file_edit
importlib.reload(tools.file_edit)
from tools.file_edit import FileEditTool
from tools.base import ToolUseContext
from permissions.manager import PermissionManager, PermissionContext
from permissions.rules import PermissionMode
from pathlib import Path

edit_tool = FileEditTool()
ctx = ToolUseContext(
    permission_manager=PermissionManager(PermissionContext(mode=PermissionMode.BYPASS)),
    model=model_id,
    tools=[edit_tool],
    session_id="test",
)

tmp = Path("../tests/_tmp_edit_test.txt")

# 场景 1：创建文件（old_string 为空）
async for msg in edit_tool.execute({"file_path": str(tmp), "old_string": "", "new_string": "hello\nworld\n"}, ctx):
    print(msg, end="")
print("内容:", tmp.read_text())

# 场景 2：正常替换
async for msg in edit_tool.execute({"file_path": str(tmp), "old_string": "world", "new_string": "claude"}, ctx):
    print(msg, end="")
print("内容:", tmp.read_text())

# 场景 3：old_string 不存在
async for msg in edit_tool.execute({"file_path": str(tmp), "old_string": "不存在的文本", "new_string": "xxx"}, ctx):
    print(msg, end="")

# 场景 4：多处匹配但未设 replace_all
tmp.write_text("aaa bbb aaa\n")
async for msg in edit_tool.execute({"file_path": str(tmp), "old_string": "aaa", "new_string": "zzz"}, ctx):
    print(msg, end="")

# 场景 5：replace_all=True
async for msg in edit_tool.execute({"file_path": str(tmp), "old_string": "aaa", "new_string": "zzz", "replace_all": True}, ctx):
    print(msg, end="")
print("内容:", tmp.read_text())

# 清理临时文件
tmp.unlink(missing_ok=True)
print("临时文件已清理")

已覆盖文件：..\tests\_tmp_edit_test.txt
内容: hello
world

已编辑 ..\tests\_tmp_edit_test.txt，替换了 1 处匹配。
内容: hello
claude

未找到指定文本，编辑失败。
请确认 old_string 与文件内容完全一致（含空格、缩进、换行）。
找到 2 处匹配，编辑失败。
old_string 必须在文件中唯一，或设置 replace_all=true。
已编辑 ..\tests\_tmp_edit_test.txt，替换了 2 处匹配。
内容: zzz bbb zzz

临时文件已清理
